# 2 · Bands, filters and SVO

A band name has to become an effective wavelength and a zero point before any photometry is
possible. WHISPER resolves names from a built-in table first, then falls back to the
[SVO Filter Profile Service](http://svo2.cab.inta-csic.es/theory/fps/), and lets you register
anything it does not know. The physical models go one step further and integrate over the band's
transmission curve; the last section shows which curve a label names.

**Needs:** the base install. The SVO fallback needs `pyphot` or `astroquery` and a network
connection; the built-in table does not.

In [1]:
from pathlib import Path

# Works whether the notebook is run from notebooks/ or from the repository root.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tests" / "data").is_dir())
AT2017GFO = ROOT / "tests" / "data" / "at2017gfo.csv"

import warnings
warnings.filterwarnings("ignore")           # keep the tutorial output readable

import numpy as np
import matplotlib.pyplot as plt
import whisper_cbpf as wp

print("whisper_cbpf", wp.__version__)

whisper_cbpf 0.2.0


## The built-in table

In [2]:
print("known bands:", len(wp.FILTER_LOOKUP))
print(sorted(wp.FILTER_LOOKUP)[:24], "...")

known bands: 97
['2massh', '2massj', '2massks', 'B', 'F070W', 'F090W', 'F110W', 'F115W', 'F125W', 'F150W', 'F160W', 'F200W', 'F277W', 'F336W', 'F356W', 'F444W', 'F475W', 'F606W', 'F625W', 'F775W', 'F814W', 'F850W', 'G', 'H'] ...


In [3]:
for k, v in wp.resolve_band("g").items():
    print(f"  {k:14s} {v}")

  group          g-band
  lambda_eff     4866.0
  zero_point     3631.0
  filter_id      LSST/LSST.g
  source         lsst


`resolve_bands` does many at once, and `group_bands` collects the points of a light curve by band —
which is what every per-band plot and metric is built on.

In [4]:
# resolve_bands is the vectorised form: arrays aligned with the input, plus the full info dict.
lam, zp, info = wp.resolve_bands(["g", "r", "ztfg", "g"])
print("lambda_eff:", np.round(lam, 1))
print("zero point:", zp)
print()
for band, meta in info.items():
    print(f"  {band:6s} lambda_eff = {meta['lambda_eff']:8.1f} A   "
          f"zp = {meta['zero_point']:.1f} Jy   via {meta['source']}")

lambda_eff: [4866. 6215. 4866. 4866.]
zero point: [3631. 3631. 3631. 3631.]

  g      lambda_eff =   4866.0 A   zp = 3631.0 Jy   via lsst
  r      lambda_eff =   6215.0 A   zp = 3631.0 Jy   via lsst
  ztfg   lambda_eff =   4866.0 A   zp = 3631.0 Jy   via lsst


In [5]:
lc = wp.load_lightcurve(AT2017GFO, redshift=0.0098)

# group_bands maps each band label to its effective broadband, so `ztfg`, `g-ztf` and `g` all
# collapse to one group and per-band plots and metrics do not split the same filter three ways.
groups = wp.group_bands(lc.bands)
for raw, grp in sorted(set(zip(lc.bands, groups)))[:8]:
    print(f"  {raw:8s} -> {grp}")

  B        -> g-band
  C        -> C
  F110W    -> J-band
  F160W    -> H-band
  F336W    -> U-band
  F475W    -> g-band
  F606W    -> r-band
  F625W    -> r-band


## LSST

`LSST_BAND_INFO` holds the effective wavelength of each group: LSST's for the `u g r i z y`
names, 2MASS's for J, H and K. The survey presets of `load_lightcurve` write the survey's
own labels, `SURVEY_BANDS`, so a model never has to guess which g a light curve means.

In [6]:
for band, meta in wp.LSST_BAND_INFO.items():
    print(f"  {band:8s} lambda_eff = {meta['lambda_eff']:7.1f} A   zp = {meta['zero_point']:.1f} Jy")

print()
print("labels the survey presets write:", wp.io.SURVEY_BANDS)

  U-band   lambda_eff =  3671.0 A   zp = 3631.0 Jy
  g-band   lambda_eff =  4866.0 A   zp = 3631.0 Jy
  r-band   lambda_eff =  6215.0 A   zp = 3631.0 Jy
  i-band   lambda_eff =  7545.0 A   zp = 3631.0 Jy
  z-band   lambda_eff =  8679.0 A   zp = 3631.0 Jy
  y-band   lambda_eff =  9710.0 A   zp = 3631.0 Jy
  J-band   lambda_eff = 12350.0 A   zp = 3631.0 Jy
  H-band   lambda_eff = 16620.0 A   zp = 3631.0 Jy
  K-band   lambda_eff = 21590.0 A   zp = 3631.0 Jy
  F356W-band lambda_eff = 35690.0 A   zp = 3631.0 Jy
  F444W-band lambda_eff = 44040.0 A   zp = 3631.0 Jy

labels the survey presets write: {'ztf': ('ztfg', 'ztfr', 'ztfi'), 'lsst': ('lsstu', 'lsstg', 'lsstr', 'lssti', 'lsstz', 'lssty')}


## Registering a band WHISPER does not know

If your filter is not in the table and not in SVO — a custom instrument, say — register it. The
entry is process-local and takes effect immediately.

In [7]:
wp.register_manual_band("myfilter", lambda_eff=6200.0, zero_point=3631.0)
print("resolved:", wp.resolve_band("myfilter"))

wp.unregister_manual_band("myfilter")
print("after unregister, still known?:", "myfilter" in wp.FILTER_LOOKUP)

wp.clear_manual_bands()   # drops every manual entry at once

resolved: {'group': 'myfilter', 'lambda_eff': 6200.0, 'zero_point': 3631.0, 'filter_id': None, 'source': 'manual'}
after unregister, still known?: False


## When resolution fails

A band that is in neither the table nor SVO comes back **unresolved** rather than guessed — a
guessed wavelength would put the flux in the wrong place and only ever show up as a biased fit.
`sdssg` is a good example: it is a real filter, but the built-in table reads bare letters as LSST
and has no SDSS entry, so without the SVO fallback it does not resolve here.

In [8]:
for band in ["g", "sdssg", "definitely-not-a-filter"]:
    meta = wp.resolve_band(band, svo_fallback=False)
    print(f"  {band:24s} source={meta['source']:12s} lambda_eff={meta['lambda_eff']}")

  g                        source=lsst         lambda_eff=4866.0
  sdssg                    source=unresolved   lambda_eff=None
  definitely-not-a-filter  source=unresolved   lambda_eff=None


In [9]:
# The SVO fallback needs astroquery and a network connection; without either it says so.
from whisper_cbpf.io.svo import resolve_band_svo

try:
    print(resolve_band_svo("definitely-not-a-filter"))
except wp.SvoUnavailable as exc:
    print("SvoUnavailable:", str(exc)[:200])
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc)[:200]}")

SvoUnavailable: No documented SVO filter ID for band 'definitely-not-a-filter' and no wavelength hint to search with. Pass register_manual_band(...) or a lambda_eff hint.


## The filter a physical model integrates

The kilonova, TDE, supernova and redback models do not use an effective wavelength: they
integrate the model spectrum over the band's transmission curve. `resolve_filter` is the one map
from a label to that curve, the same on the CPU and the GPU.

A bare `u g r i z y` names no survey, so it is read as **LSST**. Survey labels (`ztfg`, `sdssr`,
`lssty`) are unaffected. AT2017GFO's bare `g r i` are SDSS photometry, so a physical fit to them
says so: `default_system="sdss"` on the model, or `wp.set_default_band_system("sdss")` for the
session. Notebook 6 uses the explicit labels `sdssg` and `sdssr` instead.

In [10]:
print("bare letters are read as:", wp.default_band_system())
for label in ["g", "ztfg", "sdssr", "lssty"]:
    print(f"  {label:6s} -> {wp.resolve_filter(label)}")

# AT2017GFO's g is SDSS g: say so per call (or per model, with the same keyword).
print()
print(f"  {'g':6s} -> {wp.resolve_filter('g', default_system='sdss')}   (default_system='sdss')")

bare letters are read as: lsst
  g      -> lsstg
  ztfg   -> ztfg
  sdssr  -> sdssr
  lssty  -> lssty

  g      -> sdssg   (default_system='sdss')


## Where next

* [03 · Models](03_models.ipynb) — the models you fit.
* [10 · Plotting](10_plotting.ipynb) — where band colours and grouping show up.
* [`docs/PHOTOMETRY.md`](../docs/PHOTOMETRY.md) — how a model spectrum becomes a band magnitude.